# TP — Lévitation magnétique : backstepping

**Commande non linéaire — Master 3EA, parcours ASI — Université Claude Bernard Lyon 1**

Ce notebook accompagne l'énoncé `TP_Levitation.pdf`. Il **est** votre rapport : répondez aux questions dans les cellules de texte, commentez chaque figure.

> **Sur Colab :** commencez par *Fichier > Enregistrer une copie dans Drive*.

Les cellules marquées `# À COMPLÉTER` sont à écrire.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from scipy.integrate import solve_ivp
from scipy.signal import place_poles

np.set_printoptions(precision=4, suppress=True)
plt.rcParams['figure.figsize'] = (11, 3.4)

## 0. Le modèle (fourni)

$y$ : entrefer entre l'électroaimant et la bille (compté vers le bas), $v=\dot y$, $i$ : courant, $u$ : tension.

$$\dot y = v,\qquad \dot v = g - \frac{k}{m}\,\frac{i^2}{y^2},\qquad L\,\dot i = -R\,i + u .$$

La bille repose sur un support en $y = 16$ mm et ne peut pas dépasser la face de l'aimant ($y = 2$ mm). La tension est limitée à $|u|\le 15$ V.

In [ ]:
m, g, k = 0.02, 9.81, 2.5e-5        # kg, m/s², N.m²/A²
R, L = 10.0, 0.3                     # ohm, H
umax = 15.0                          # V
y_sup, y_min = 0.016, 0.002          # support, face de l'aimant (m)

def courant_eq(yr, masse=m):
    """Courant d'équilibre pour un entrefer yr."""
    return yr * np.sqrt(masse * g / k)

def levitation(t, x, loi, masse=m):
    """Dynamique de la bille ; loi(x) donne la tension (saturée ici)."""
    y, v, i = x
    u = np.clip(loi(x), -umax, umax)
    a = g - (k / masse) * i**2 / y**2
    if y >= y_sup and v >= 0 and a > 0:      # bille posée sur le support
        a, v = 0.0, 0.0
    if y <= y_min and v <= 0 and a < 0:      # bille collée à l'aimant
        a, v = 0.0, 0.0
    return [v, a, (-R * i + u) / L]

def simule(loi, x0, T=0.6, masse=m):
    sol = solve_ivp(levitation, (0, T), x0, args=(loi, masse), max_step=2e-4, rtol=1e-7, atol=1e-10)
    u = np.array([np.clip(loi(sol.y[:, j]), -umax, umax) for j in range(sol.y.shape[1])])
    return sol.t, sol.y, u

def trace(resultats, titre, yr=None):
    fig, ax = plt.subplots(1, 3)
    for lab, t, x, u in resultats:
        ax[0].plot(t, 1e3 * x[0], label=lab)
        ax[1].plot(t, x[2], label=lab)
        ax[2].plot(t, u, label=lab)
    if yr is not None:
        ax[0].axhline(1e3 * yr, color='k', ls=':', lw=0.8)
    ax[0].set(xlabel='t (s)', ylabel='y (mm)', title=titre); ax[0].invert_yaxis()
    ax[1].set(xlabel='t (s)', ylabel='i (A)'); ax[2].set(xlabel='t (s)', ylabel='u (V)')
    for a in ax:
        a.grid(True); a.legend(fontsize=7)
    plt.tight_layout(); plt.show()

y0 = 0.008
i0 = courant_eq(y0)
print('équilibre à 8 mm : i = %.3f A, u = %.2f V' % (i0, R * i0))

# Séance 1 — Approches linéaire et linéarisante

### Q4. Boucle ouverte
Bille à l'équilibre, tension constante $u = R\,i_0$, mais un écart de 0,1 mm sur la position initiale.

In [ ]:
loi_cste = lambda x: R * i0
trace([('y(0) = 8,1 mm', *simule(loi_cste, [0.0081, 0, i0], T=0.3)),
       ('y(0) = 7,9 mm', *simule(loi_cste, [0.0079, 0, i0], T=0.3))], 'boucle ouverte')

*Votre commentaire :*

### Q2 et Q5. Retour d'état linéaire, réglé à 8 mm

In [ ]:
def lineaire(yr):
    """Linéarisé autour de l'équilibre d'entrefer yr."""
    ir = courant_eq(yr)
    A = np.array([[0, 1, 0], [2 * g / yr, 0, -2 * g / ir], [0, 0, -R / L]])
    B = np.array([[0], [0], [1 / L]])
    return A, B

# À COMPLÉTER : linéarisé à 8 mm et gain K_lin (pôles -40, -50, -60), pour u = u_r - K (x - x_r)
A8, B8 = lineaire(y0)
K_lin = None
print('pôles en boucle ouverte :', np.linalg.eigvals(A8))
print('K_lin =', K_lin)

def loi_lin(yr):
    ir = courant_eq(yr)
    return lambda x: R * ir - float((K_lin @ (np.asarray(x) - np.array([yr, 0, ir])))[0])

In [ ]:
trace([('8 -> 5 mm', *simule(loi_lin(0.005), [0.008, 0, i0])),
       ('8 -> 12 mm', *simule(loi_lin(0.012), [0.008, 0, i0])),
       ('décollage 16 -> 8 mm', *simule(loi_lin(0.008), [y_sup, 0, 0.0]))], 'retour linéaire réglé à 8 mm')

Le même gain, utilisé à d'autres entrefers : où vont les pôles de la boucle fermée linéarisée ?

In [ ]:
# À COMPLÉTER : pôles de A(y_r) - B K_lin pour y_r = 3, 5, 8, 12, 15 mm

*Votre commentaire :*

### Q3 et Q6. Linéarisation par bouclage
On impose $\dddot y = -c_2\ddot y - c_1\dot y - c_0(y-y_r)$ avec $(s+\lambda)^3$, $\lambda = 40$ rad/s.

In [ ]:
lam = 40.0
c2, c1, c0 = 3 * lam, 3 * lam**2, lam**3

def loi_fl(yr, masse_modele=m):
    kk = k / masse_modele
    def loi(x):
        y, v, i = x
        # À COMPLÉTER : y''' = a(x) + b(x) u ; renvoyer u = (w - a) / b
        return 0.0
    return loi

trace([('8 -> 5 mm', *simule(loi_fl(0.005), [0.008, 0, i0])),
       ('8 -> 12 mm', *simule(loi_fl(0.012), [0.008, 0, i0]))], 'linéarisation par bouclage')

Décollage : la bille est posée sur le support, bobine éteinte ($i = 0$). Que vaut $b(x)$ ?

In [ ]:
try:
    print('u =', loi_fl(0.008)([y_sup, 0.0, 0.0]))
except ZeroDivisionError as err:
    print('Échec du calcul de la commande :', err)
for i_test in (1e-1, 1e-2, 1e-3):
    print('i = %.0e A  ->  u demandée = %.3g V' % (i_test, loi_fl(0.008)([y_sup, 0.0, i_test])))

*Votre commentaire :*

### Q6 (suite). Erreur sur la masse
La bille réelle est 30 % plus lourde (ou plus légère) que celle du modèle.

In [ ]:
for ratio in (0.7, 1.0, 1.3):
    t, x, u = simule(loi_fl(0.008), [0.009, 0, courant_eq(0.009, ratio * m)], T=1.0, masse=ratio * m)
    print('masse réelle = %.1f m : y final = %.2f mm (consigne 8 mm)' % (ratio, 1e3 * x[0, -1]))

*Votre commentaire :*

# Séance 2 — Backstepping

### Q9. La loi de backstepping
Implémentez la loi obtenue à la question 7, avec les notations $e_1 = y - y_r$, $e_2 = v + k_1 e_1$, $w_d$, $i_d$, $e_3 = i - i_d$.

Les états mesurés donnent l'accélération $a = g - (k/m)\,i^2/y^2$ : utilisez-la pour calculer $\dot w_d$ puis $\dot i_d$ sans dériver numériquement.

La troncature de $w_d$ à $w_{\min}$ (question 8) est faite par la fonction `tronque`, fournie : elle est lissée, pour que la tension demandée reste continue. Une troncature brutale `max(wd, w_min)` rend $\dot i_d$, donc $u$, discontinue, et la boucle fermée peut alors brouter sur la surface $w_d=w_{\min}$.

In [ ]:
def tronque(wd, wd_point, w_min, delta=0.05 * g):
    """Troncature lissée de w_d à w_min (fournie). Renvoie la valeur tronquée et sa dérivée.
    Égale à w_d dès que w_d >= w_min + delta, à w_min dès que w_d <= w_min - delta, raccord C1 entre les deux."""
    d = wd - w_min
    if d >= delta:
        return wd, wd_point
    if d <= -delta:
        return w_min, 0.0
    return w_min + (d + delta)**2 / (4 * delta), (d + delta) / (2 * delta) * wd_point

def loi_bs(yr, k1=40.0, k2=40.0, k3=40.0, masse_modele=m, w_min=0.2 * g):
    kk = k / masse_modele
    def loi(x):
        y, v, i = x
        e1 = y - yr
        e2 = v + k1 * e1
        a = g - kk * i**2 / y**2
        # À COMPLÉTER : w_d et sa dérivée, troncature avec tronque(wd, wd_point, w_min),
        # puis i_d, sa dérivée, e3 et u
        return 0.0
    return loi

trace([('8 -> 5 mm', *simule(loi_bs(0.005), [0.008, 0, i0])),
       ('8 -> 12 mm', *simule(loi_bs(0.012), [0.008, 0, i0])),
       ('décollage 16 -> 8 mm', *simule(loi_bs(0.008), [y_sup, 0, 0.0]))], 'backstepping')

### Q10. La même réponse à tous les entrefers ?
On trace l'erreur normalisée $(y - y_r)/(y(0) - y_r)$ pour des sauts de $\pm1$ mm autour de plusieurs entrefers.

In [ ]:
fig, ax = plt.subplots(1, 2)
for nom, fabrique, a_ in (('linéaire', loi_lin, ax[0]), ('backstepping', loi_bs, ax[1])):
    for yr in (0.004, 0.008, 0.013):
        y_init = yr + 0.001
        t, x, u = simule(fabrique(yr), [y_init, 0, courant_eq(y_init)], T=0.25)
        a_.plot(t, (x[0] - yr) / (y_init - yr), label='y_r = %.0f mm' % (1e3 * yr))
    a_.set(title=nom, xlabel='t (s)', ylabel='erreur normalisée'); a_.grid(True); a_.legend(fontsize=8)
plt.tight_layout(); plt.show()

*Votre commentaire :*

### Q11. Erreur sur la masse, et action intégrale

In [ ]:
for ratio in (0.7, 1.0, 1.3):
    t, x, u = simule(loi_bs(0.008), [0.009, 0, courant_eq(0.009, ratio * m)], T=1.0, masse=ratio * m)
    print('backstepping, masse réelle = %.1f m : y final = %.2f mm' % (ratio, 1e3 * x[0, -1]))

On ajoute l'état $z$ avec $\dot z = y - y_r$, en tête de chaîne : $(z, y, v, i)$. C'est une étape de backstepping de plus. Pour ne pas dériver à la main, on laisse `sympy` calculer la commande virtuelle $w_d(z,y,v)$ et ses dérivées partielles.

In [ ]:
z_, y_, v_, yr_, k0_, k1_, k2_ = sp.symbols('z y v y_r k0 k1 k2', real=True)
# À COMPLÉTER : E1, v_d, E2 et w_d(z, y, v) pour la chaîne (z, y, v, i)
E1, E2, wd_expr = None, None, None
print('w_d =', sp.collect(wd_expr, [z_, y_, v_]))
arguments = (z_, y_, v_, yr_, k0_, k1_, k2_)
f_wd = sp.lambdify(arguments, wd_expr)
f_grad = sp.lambdify(arguments, [sp.diff(wd_expr, s) for s in (z_, y_, v_)])
f_e2 = sp.lambdify(arguments, E2)

In [ ]:
def loi_bs_int(yr, k0=15.0, k1=40.0, k2=40.0, k3=40.0, masse_modele=m, w_min=0.2 * g):
    kk = k / masse_modele
    def loi(X):
        z, y, v, i = X
        p = (z, y, v, yr, k0, k1, k2)
        a = g - kk * i**2 / y**2
        gz, gy, gv = f_grad(*p)
        wd, wd_point = tronque(f_wd(*p), gz * (y - yr) + gy * v + gv * a, w_min)
        # À COMPLÉTER : même fin que dans loi_bs, avec e2 = f_e2(*p)
        return 0.0
    return loi

def simule_int(loi, X0, yr, T=1.5, masse=m):
    def f(t, X):
        u = np.clip(loi(X), -umax, umax)
        return [X[1] - yr, *levitation(t, X[1:], lambda x: u, masse)]
    sol = solve_ivp(f, (0, T), X0, max_step=2e-4, rtol=1e-7, atol=1e-10)
    return sol.t, sol.y

plt.figure()
for ratio in (0.7, 1.0, 1.3):
    t, X = simule_int(loi_bs_int(0.008), [0, 0.009, 0, courant_eq(0.009, ratio * m)], 0.008, masse=ratio * m)
    plt.plot(t, 1e3 * X[1], label='masse réelle = %.1f m' % ratio)
    print('backstepping + intégrale, masse %.1f m : y final = %.3f mm' % (ratio, 1e3 * X[1, -1]))
plt.gca().invert_yaxis(); plt.xlabel('t (s)'); plt.ylabel('y (mm)'); plt.grid(True); plt.legend(); plt.title('backstepping avec action intégrale'); plt.show()

*Votre commentaire :*

### Question ouverte (une au choix, obligatoire)
Choisissez une des trois questions de l'énoncé (explosion de complexité, saturation de la tension, vitesse non mesurée) et traitez-la ici. Elle n'est pas guidée et doit apparaître dans la soutenance.

In [ ]:
# Votre travail